# Loan Approval Prediction

## 1. Imports and Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # for chart saving
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, json, os, warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, roc_curve,
                             confusion_matrix, classification_report)

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

sns.set_theme(style='whitegrid')
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'figure.dpi': 150,
    'savefig.dpi': 150,
    'savefig.bbox': 'tight',
    'font.family': 'sans-serif',
})

os.makedirs('charts', exist_ok=True)
os.makedirs('models', exist_ok=True)

print('Setup complete — charts/ and models/ directories ready.')


Setup complete — charts/ and models/ directories ready.


## 2. Data Loading & Initial Exploration

In [2]:
df = pd.read_csv('data/loan_data.csv')
print(f'Dataset shape: {df.shape}')
display(df.head())


Dataset shape: (45000, 14)


,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
0,22.0,female,Master,71948.0,0,RENT,35000.0,PERSONAL,16.02,0.49,3.0,561,No,1
1,21.0,female,High School,12282.0,0,OWN,1000.0,EDUCATION,11.14,0.08,2.0,504,Yes,0
2,25.0,female,High School,12438.0,3,MORTGAGE,5500.0,MEDICAL,12.87,0.44,3.0,635,No,1
3,23.0,female,Bachelor,79753.0,0,RENT,35000.0,MEDICAL,15.23,0.44,2.0,675,No,1
4,24.0,male,Master,66135.0,1,RENT,35000.0,MEDICAL,14.27,0.53,4.0,586,No,1


In [3]:
# Dataset shape, types, and missing value summary
display(df.info())
print('\nMissing values:\n')
display(df.isnull().sum())
print('\nBasic statistics:\n')
display(df.describe())


<class 'pandas.DataFrame'>
RangeIndex: 45000 entries, 0 to 44999
Data columns (total 14 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   person_age                      45000 non-null  float64
 1   person_gender                   45000 non-null  str    
 2   person_education                45000 non-null  str    
 3   person_income                   45000 non-null  float64
 4   person_emp_exp                  45000 non-null  int64  
 5   person_home_ownership           45000 non-null  str    
 6   loan_amnt                       45000 non-null  float64
 7   loan_intent                     45000 non-null  str    
 8   loan_int_rate                   45000 non-null  float64
 9   loan_percent_income             45000 non-null  float64
 10  cb_person_cred_hist_length      45000 non-null  float64
 11  credit_score                    45000 non-null  int64  
 12  previous_loan_defaults_on_file  45000 non-n

None


Missing values:



person_age                        0
person_gender                     0
person_education                  0
person_income                     0
person_emp_exp                    0
person_home_ownership             0
loan_amnt                         0
loan_intent                       0
loan_int_rate                     0
loan_percent_income               0
cb_person_cred_hist_length        0
credit_score                      0
previous_loan_defaults_on_file    0
loan_status                       0
dtype: int64


Basic statistics:



,person_age,person_income,person_emp_exp,loan_amnt,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,loan_status
count,45000.000000,4.500000e+04,45000.000000,45000.000000,45000.000000,45000.000000,45000.000000,45000.000000,45000.000000
mean,27.764178,8.031905e+04,5.410333,9583.157556,11.006606,0.139725,5.867489,632.608756,0.222222
std,6.045108,8.042250e+04,6.063532,6314.886691,2.978808,0.087212,3.879702,50.435865,0.415744
min,20.000000,8.000000e+03,0.000000,500.000000,5.420000,0.000000,2.000000,390.000000,0.000000
25%,24.000000,4.720400e+04,1.000000,5000.000000,8.590000,0.070000,3.000000,601.000000,0.000000
50%,26.000000,6.704800e+04,4.000000,8000.000000,11.010000,0.120000,4.000000,640.000000,0.000000
75%,30.000000,9.578925e+04,8.000000,12237.250000,12.990000,0.190000,8.000000,670.000000,0.000000
max,144.000000,7.200766e+06,125.000000,35000.000000,20.000000,0.660000,30.000000,850.000000,1.000000


## 3. Exploratory Data Analysis (EDA)
Distribution, correlation, and outlier analysis for numerical and categorical
features. All charts are saved to `charts/`.

In [4]:
# Target variable class distribution
fig, ax = plt.subplots(figsize=(6, 4))
counts = df['loan_status'].value_counts()
colors = ['#4f7cff', '#7c5cfc']
bars = ax.bar(counts.index.map({0: 'Rejected', 1: 'Approved'}),
              counts.values, color=colors, width=0.5,
              edgecolor='white', linewidth=0.8)

for b in bars:
    ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 200,
            f'{b.get_height():,}', ha='center', fontweight='bold', fontsize=11)

ax.set_title('Target Variable Distribution (loan_status)', fontweight='bold', fontsize=13)
ax.set_ylabel('Count')
plt.savefig('charts/target_distribution.png')
plt.show()

print(f'Class ratio — Approved: {counts.get(1, 0)}, Rejected: {counts.get(0, 0)}')


Class ratio — Approved: 10000, Rejected: 35000


In [5]:
# Numerical feature distributions with KDE overlay
num_cols = ['person_age', 'person_income', 'loan_amnt', 'loan_int_rate',
            'credit_score', 'person_emp_exp', 'loan_percent_income', 'cb_person_cred_hist_length']

fig, axes = plt.subplots(2, 4, figsize=(20, 8))
palette = ['#4f7cff', '#7c5cfc', '#10b981', '#f59e0b',
           '#ef4444', '#ec4899', '#06b6d4', '#8b5cf6']

for i, col in enumerate(num_cols):
    ax = axes[i // 4][i % 4]
    sns.histplot(df[col], bins=30, ax=ax, color=palette[i], kde=True,
                 edgecolor='white', linewidth=0.3)
    ax.set_title(col, fontweight='bold', fontsize=10)
    ax.set_xlabel('')

fig.suptitle('Numerical Feature Distributions', fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('charts/feature_distributions.png')
plt.show()


In [6]:
# Outlier analysis via boxplots for key continuous features
outlier_cols = ['person_age', 'person_income', 'person_emp_exp',
                'loan_amnt', 'loan_int_rate', 'credit_score']

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for i, col in enumerate(outlier_cols):
    ax = axes[i // 3][i % 3]
    sns.boxplot(data=df, y=col, ax=ax, color='#4f7cff', width=0.4)
    ax.set_title(col, fontweight='bold', fontsize=10)

plt.suptitle('Outlier Analysis — Boxplots', fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('charts/outlier_boxplots.png')
plt.show()


In [7]:
# Feature correlation heatmap (lower triangle)
fig, ax = plt.subplots(figsize=(10, 8))
corr = df.select_dtypes(include=[np.number]).corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.5, ax=ax, square=True, cbar_kws={'shrink': 0.8})

ax.set_title('Feature Correlation Heatmap', fontweight='bold', fontsize=14)
plt.savefig('charts/correlation_heatmap.png')
plt.show()


In [8]:
# Categorical feature distributions by loan status
cat_cols = ['person_gender', 'person_education', 'person_home_ownership',
            'loan_intent', 'previous_loan_defaults_on_file']

fig, axes = plt.subplots(1, 5, figsize=(22, 4))

for i, col in enumerate(cat_cols):
    sns.countplot(data=df, x=col, hue='loan_status', ax=axes[i],
                  palette=['#4f7cff', '#10b981'])
    axes[i].set_title(col, fontweight='bold', fontsize=9)
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=35, labelsize=7)
    if i > 0:
        axes[i].get_legend().remove()
    else:
        axes[i].legend(title='Status', labels=['Rejected', 'Approved'], fontsize=7)

fig.suptitle('Categorical Features vs Loan Status', fontweight='bold', fontsize=14, y=1.05)
plt.tight_layout()
plt.savefig('charts/categorical_distributions.png')
plt.show()


In [9]:
# Income vs loan amount, colored by loan status
fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(df['person_income'], df['loan_amnt'],
                     c=df['loan_status'], cmap='coolwarm', alpha=0.3,
                     s=10, edgecolors='none')
ax.set_xlim(0, 300000)
ax.set_xlabel('Annual Income ($)', fontweight='bold')
ax.set_ylabel('Loan Amount ($)', fontweight='bold')
ax.set_title('Income vs Loan Amount by Approval Status', fontweight='bold', fontsize=14)
ax.legend(*scatter.legend_elements(), title='Status', labels=['Rejected', 'Approved'])
ax.grid(alpha=0.3)
plt.savefig('charts/income_vs_loan.png')
plt.show()


## 4. Data Preprocessing Pipeline
A `ColumnTransformer` with three sub-pipelines handles each feature type:
numerical features are imputed (median) and scaled (`StandardScaler`);
education is ordinal-encoded to preserve its hierarchy, while remaining categoricals
are one-hot encoded. The entire transform is encapsulated in a pipeline to
prevent data leakage during cross-validation.

In [10]:
# Feature-target separation
X = df.drop('loan_status', axis=1)
y = df['loan_status']

# Feature categorization by data type
numerical_features = ['person_age', 'person_income', 'person_emp_exp', 'loan_amnt',
                      'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length', 'credit_score']

ordinal_features = ['person_education']
education_order = [['High School', 'Associate', 'Bachelor', 'Master', 'Doctorate']]

nominal_features = ['person_gender', 'person_home_ownership', 'loan_intent', 'previous_loan_defaults_on_file']

# Sub-pipelines: imputation + transformation per feature type
num_pipe = ImbPipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

ord_pipe = ImbPipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(categories=education_order, handle_unknown='use_encoded_value', unknown_value=-1))
])

nom_pipe = ImbPipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

# Combined preprocessor for all feature types
preprocessor = ColumnTransformer([
    ('num', num_pipe, numerical_features),
    ('ord', ord_pipe, ordinal_features),
    ('nom', nom_pipe, nominal_features)
])

print('Preprocessor defined.')


Preprocessor defined.


## 5. Train-Test Split
80/20 stratified split to preserve the class distribution across both sets.

In [11]:
# 80/20 stratified train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')


Train: (36000, 13)  |  Test: (9000, 13)


## 6. Baseline Model & SMOTE Impact
Logistic Regression serves as the baseline. To justify the use of SMOTE, the
baseline is trained both with and without oversampling.

In [12]:
def evaluate_model(name, pipeline, X_tr, y_tr, X_te, y_te):
    pipeline.fit(X_tr, y_tr)
    y_pred = pipeline.predict(X_te)
    y_prob = pipeline.predict_proba(X_te)[:, 1]

    metrics = {
        'accuracy':  round(accuracy_score(y_te, y_pred), 4),
        'precision': round(precision_score(y_te, y_pred), 4),
        'recall':    round(recall_score(y_te, y_pred), 4),
        'f1':        round(f1_score(y_te, y_pred), 4),
        'roc_auc':   round(roc_auc_score(y_te, y_prob), 4),
    }

    print(f'--- {name} ---')
    for k, v in metrics.items():
        print(f'  {k:>10s}: {v}')
    print('\n' + classification_report(y_te, y_pred, target_names=['Rejected', 'Approved']))
    print('-' * 40)

    return pipeline, y_pred, y_prob, metrics


In [13]:
# SMOTE impact comparison: Logistic Regression with vs without oversampling

# Without SMOTE
pipe_no_smote = ImbPipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])
_, y_pred_ns, y_prob_ns, metrics_ns = evaluate_model(
    'Logistic Regression (no SMOTE)', pipe_no_smote,
    X_train, y_train, X_test, y_test)

# With SMOTE
pipe_smote = ImbPipeline([
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])
_, y_pred_s, y_prob_s, metrics_s = evaluate_model(
    'Logistic Regression (with SMOTE)', pipe_smote,
    X_train, y_train, X_test, y_test)

print(f'\nRecall improvement with SMOTE: {metrics_ns["recall"]:.4f} -> {metrics_s["recall"]:.4f}')


--- Logistic Regression (no SMOTE) ---
    accuracy: 0.8994
   precision: 0.7892
      recall: 0.747
          f1: 0.7675
     roc_auc: 0.9562

              precision    recall  f1-score   support

    Rejected       0.93      0.94      0.94      7000
    Approved       0.79      0.75      0.77      2000

    accuracy                           0.90      9000
   macro avg       0.86      0.84      0.85      9000
weighted avg       0.90      0.90      0.90      9000

----------------------------------------
--- Logistic Regression (with SMOTE) ---
    accuracy: 0.8622
   precision: 0.6318
      recall: 0.911
          f1: 0.7461
     roc_auc: 0.9562

              precision    recall  f1-score   support

    Rejected       0.97      0.85      0.91      7000
    Approved       0.63      0.91      0.75      2000

    accuracy                           0.86      9000
   macro avg       0.80      0.88      0.83      9000
weighted avg       0.90      0.86      0.87      9000

---------------

## 7. Comparative Modeling
All five classifiers listed in the proposal are trained with SMOTE-augmented pipelines.

In [14]:
# Classifier definitions (all five from the proposal)
classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree':       DecisionTreeClassifier(random_state=42),
    'Random Forest':       RandomForestClassifier(random_state=42),
    'SVM':                 SVC(probability=True, random_state=42),
    'KNN':                 KNeighborsClassifier(),
}

results = {}  # name -> {pipeline, y_pred, y_prob, metrics}

for name, clf in classifiers.items():
    pipe = ImbPipeline([
        ('preprocessor', preprocessor),
        ('smote', SMOTE(random_state=42)),
        ('classifier', clf)
    ])
    pipeline, y_pred, y_prob, metrics = evaluate_model(
        name, pipe, X_train, y_train, X_test, y_test)
    results[name] = {
        'pipeline': pipeline, 'y_pred': y_pred,
        'y_prob': y_prob, 'metrics': metrics
    }

print('All base models trained.')


--- Logistic Regression ---
    accuracy: 0.8622
   precision: 0.6318
      recall: 0.911
          f1: 0.7461
     roc_auc: 0.9562

              precision    recall  f1-score   support

    Rejected       0.97      0.85      0.91      7000
    Approved       0.63      0.91      0.75      2000

    accuracy                           0.86      9000
   macro avg       0.80      0.88      0.83      9000
weighted avg       0.90      0.86      0.87      9000

----------------------------------------
--- Decision Tree ---
    accuracy: 0.8892
   precision: 0.725
      recall: 0.808
          f1: 0.7642
     roc_auc: 0.8602

              precision    recall  f1-score   support

    Rejected       0.94      0.91      0.93      7000
    Approved       0.72      0.81      0.76      2000

    accuracy                           0.89      9000
   macro avg       0.83      0.86      0.85      9000
weighted avg       0.89      0.89      0.89      9000

----------------------------------------
--- R

## 8. Hyperparameter Tuning
Grid search with Stratified 5-Fold CV, scored on ROC-AUC. ROC-AUC is chosen
because accuracy alone can be misleading under class imbalance (Lessmann et
al., 2015). Models tuned: Random Forest, SVM (as specified in the proposal),
and KNN.

In [15]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Random Forest tuning
rf_params = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [None, 10, 20],
    'classifier__min_samples_split': [2, 5],
}

rf_pipe = ImbPipeline([
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('classifier', RandomForestClassifier(random_state=42))
])

print('Tuning Random Forest...')
rf_grid = GridSearchCV(rf_pipe, rf_params, scoring='roc_auc', cv=cv, n_jobs=-1, verbose=1)
rf_grid.fit(X_train, y_train)

print(f'Best params: {rf_grid.best_params_}')
print(f'Best CV ROC-AUC: {rf_grid.best_score_:.4f} ± {rf_grid.cv_results_["std_test_score"][rf_grid.best_index_]:.4f}\n')

best_rf = rf_grid.best_estimator_
y_pred_rf = best_rf.predict(X_test)
y_prob_rf = best_rf.predict_proba(X_test)[:, 1]

rf_metrics = {
    'accuracy':  round(accuracy_score(y_test, y_pred_rf), 4),
    'precision': round(precision_score(y_test, y_pred_rf), 4),
    'recall':    round(recall_score(y_test, y_pred_rf), 4),
    'f1':        round(f1_score(y_test, y_pred_rf), 4),
    'roc_auc':   round(roc_auc_score(y_test, y_prob_rf), 4),
}

results['Random Forest (Tuned)'] = {
    'pipeline': best_rf, 'y_pred': y_pred_rf,
    'y_prob': y_prob_rf, 'metrics': rf_metrics
}

for k, v in rf_metrics.items():
    print(f'  {k:>10s}: {v}')


Tuning Random Forest...
Fitting 5 folds for each of 12 candidates, totalling 60 fits
Best params: {'classifier__max_depth': None, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 200}
Best CV ROC-AUC: 0.9723 ± 0.0016

    accuracy: 0.9218
   precision: 0.8176
      recall: 0.834
          f1: 0.8257
     roc_auc: 0.9736


In [16]:
# SVM tuning (C and kernel, as specified in proposal)
svm_params = {
    'classifier__C': [0.1, 1, 10],
    'classifier__kernel': ['rbf', 'linear'],
}

svm_pipe = ImbPipeline([
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('classifier', SVC(probability=True, random_state=42))
])

print('Tuning SVM...')
svm_grid = GridSearchCV(svm_pipe, svm_params, scoring='roc_auc', cv=cv, n_jobs=-1, verbose=1)
svm_grid.fit(X_train, y_train)

print(f'Best params: {svm_grid.best_params_}')
print(f'Best CV ROC-AUC: {svm_grid.best_score_:.4f} ± {svm_grid.cv_results_["std_test_score"][svm_grid.best_index_]:.4f}\n')

best_svm = svm_grid.best_estimator_
y_pred_svm = best_svm.predict(X_test)
y_prob_svm = best_svm.predict_proba(X_test)[:, 1]

svm_metrics = {
    'accuracy':  round(accuracy_score(y_test, y_pred_svm), 4),
    'precision': round(precision_score(y_test, y_pred_svm), 4),
    'recall':    round(recall_score(y_test, y_pred_svm), 4),
    'f1':        round(f1_score(y_test, y_pred_svm), 4),
    'roc_auc':   round(roc_auc_score(y_test, y_prob_svm), 4),
}

results['SVM (Tuned)'] = {
    'pipeline': best_svm, 'y_pred': y_pred_svm,
    'y_prob': y_prob_svm, 'metrics': svm_metrics
}

for k, v in svm_metrics.items():
    print(f'  {k:>10s}: {v}')


Tuning SVM...
Fitting 5 folds for each of 6 candidates, totalling 30 fits
Best params: {'classifier__C': 1, 'classifier__kernel': 'rbf'}
Best CV ROC-AUC: 0.9601 ± 0.0027

    accuracy: 0.8824
   precision: 0.6728
      recall: 0.917
          f1: 0.7761
     roc_auc: 0.9616


In [17]:
# KNN tuning
knn_params = {
    'classifier__n_neighbors': [3, 5, 7, 9, 11],
    'classifier__weights': ['uniform', 'distance'],
}

knn_pipe = ImbPipeline([
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('classifier', KNeighborsClassifier())
])

print('Tuning KNN...')
knn_grid = GridSearchCV(knn_pipe, knn_params, scoring='roc_auc', cv=cv, n_jobs=-1, verbose=1)
knn_grid.fit(X_train, y_train)

print(f'Best params: {knn_grid.best_params_}')
print(f'Best CV ROC-AUC: {knn_grid.best_score_:.4f} ± {knn_grid.cv_results_["std_test_score"][knn_grid.best_index_]:.4f}\n')

best_knn = knn_grid.best_estimator_
y_pred_knn = best_knn.predict(X_test)
y_prob_knn = best_knn.predict_proba(X_test)[:, 1]

knn_metrics = {
    'accuracy':  round(accuracy_score(y_test, y_pred_knn), 4),
    'precision': round(precision_score(y_test, y_pred_knn), 4),
    'recall':    round(recall_score(y_test, y_pred_knn), 4),
    'f1':        round(f1_score(y_test, y_pred_knn), 4),
    'roc_auc':   round(roc_auc_score(y_test, y_prob_knn), 4),
}

results['KNN (Tuned)'] = {
    'pipeline': best_knn, 'y_pred': y_pred_knn,
    'y_prob': y_prob_knn, 'metrics': knn_metrics
}

for k, v in knn_metrics.items():
    print(f'  {k:>10s}: {v}')


Tuning KNN...
Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best params: {'classifier__n_neighbors': 11, 'classifier__weights': 'distance'}
Best CV ROC-AUC: 0.9346 ± 0.0023

    accuracy: 0.8474
   precision: 0.6071
      recall: 0.8885
          f1: 0.7213
     roc_auc: 0.9374


## 9. Presentation Charts
Comparative visualizations for all models. All figures are saved to `charts/`.

In [18]:
# Model performance comparison (grouped bar chart)
metric_names = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
model_names = list(results.keys())
n_models = len(model_names)

fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(metric_names))
width = 0.12
colors = ['#4f7cff', '#7c5cfc', '#10b981', '#f59e0b',
          '#ef4444', '#ec4899', '#06b6d4', '#8b5cf6']

for i, model in enumerate(model_names):
    vals = [results[model]['metrics'][m] for m in metric_names]
    offset = (i - n_models / 2 + 0.5) * width
    ax.bar(x + offset, vals, width, label=model,
           color=colors[i % len(colors)], edgecolor='white', linewidth=0.5)

ax.set_xticks(x)
ax.set_xticklabels(['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC'],
                   fontweight='bold')
ax.set_ylim(0, 1.08)
ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison', fontweight='bold', fontsize=15)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=4,
          fontsize=8, frameon=True)
ax.grid(axis='y', alpha=0.3)
plt.savefig('charts/model_comparison.png')
plt.show()


In [19]:
# Confusion matrices for all models
n = len(results)
cols = min(n, 4)
rows = (n + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(4.5 * cols, 4 * rows))
if n == 1:
    axes = np.array([axes])
axes = axes.flatten()

for i, (name, res) in enumerate(results.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=['Rejected', 'Approved'],
                yticklabels=['Rejected', 'Approved'],
                cbar=False, linewidths=1, linecolor='white')
    axes[i].set_title(name, fontweight='bold', fontsize=10)
    axes[i].set_ylabel('Actual')
    axes[i].set_xlabel('Predicted')

# Hide unused subplot slots
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Confusion Matrices', fontweight='bold', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('charts/confusion_matrices.png')
plt.show()


In [20]:
# ROC curves — all models overlaid
fig, ax = plt.subplots(figsize=(8, 7))
colors = ['#4f7cff', '#7c5cfc', '#10b981', '#f59e0b',
          '#ef4444', '#ec4899', '#06b6d4', '#8b5cf6']

for i, (name, res) in enumerate(results.items()):
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    auc_val = res['metrics']['roc_auc']
    ax.plot(fpr, tpr, color=colors[i % len(colors)], lw=2,
            label=f'{name} (AUC={auc_val:.4f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.4, label='Random Baseline')
ax.set_xlabel('False Positive Rate', fontweight='bold')
ax.set_ylabel('True Positive Rate', fontweight='bold')
ax.set_title('ROC Curves — All Models', fontweight='bold', fontsize=15)
ax.legend(loc='lower right', fontsize=8, frameon=True)
ax.grid(alpha=0.3)
plt.savefig('charts/roc_curves.png')
plt.show()


In [21]:
# Feature importance from the Tuned Random Forest
try:
    feature_names = (
        numerical_features +
        ordinal_features +
        list(results['Random Forest (Tuned)']['pipeline']
             .named_steps['preprocessor']
             .named_transformers_['nom']
             .named_steps['encoder']
             .get_feature_names_out(nominal_features))
    )
except Exception:
    feature_names = [f'feature_{i}' for i in range(
        len(results['Random Forest (Tuned)']['pipeline']
            .named_steps['classifier'].feature_importances_))]

importances = results['Random Forest (Tuned)']['pipeline'].named_steps['classifier'].feature_importances_
feat_imp = pd.Series(importances, index=feature_names).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, max(6, len(feat_imp) * 0.35)))
feat_imp.plot.barh(ax=ax, color='#4f7cff', edgecolor='white', linewidth=0.5)
ax.set_title('Feature Importance (Tuned Random Forest)', fontweight='bold', fontsize=14)
ax.set_xlabel('Importance')
ax.grid(axis='x', alpha=0.3)
plt.savefig('charts/feature_importance.png')
plt.show()


In [22]:
# Summary table with best score highlighted per metric
rows_list = []
for name, res in results.items():
    row = {'Model': name}
    row.update(res['metrics'])
    rows_list.append(row)

summary_df = pd.DataFrame(rows_list).set_index('Model')
display(summary_df.style.highlight_max(axis=0, color='#10b981').format('{:.4f}'))


,accuracy,precision,recall,f1,roc_auc
Model,,,,,
Logistic Regression,0.8622,0.6318,0.9110,0.7461,0.9562
Decision Tree,0.8892,0.7250,0.8080,0.7642,0.8602
Random Forest,0.9199,0.8152,0.8270,0.8210,0.9730
SVM,0.8824,0.6728,0.9170,0.7761,0.9616
KNN,0.8453,0.6073,0.8605,0.7120,0.9153
Random Forest (Tuned),0.9218,0.8176,0.8340,0.8257,0.9736
SVM (Tuned),0.8824,0.6728,0.9170,0.7761,0.9616
KNN (Tuned),0.8474,0.6071,0.8885,0.7213,0.9374


## 10. Export All Models
Every trained pipeline is serialized to `models/`. A `model_info.json` records
each model's metrics and flags the best performer so the Flask app can default
to it.

In [23]:
# Best model determination by ROC-AUC
best_name = max(results, key=lambda n: results[n]['metrics']['roc_auc'])
print(f'Best model: {best_name} (ROC-AUC = {results[best_name]["metrics"]["roc_auc"]})\n')

model_info = {'best_model': '', 'models': {}}

for name, res in results.items():
    safe_name = name.lower().replace(' ', '_').replace('(', '').replace(')', '')
    path = f'models/{safe_name}.pkl'
    joblib.dump(res['pipeline'], path)
    model_info['models'][safe_name] = {
        'display_name': name,
        'file': path,
        **res['metrics']
    }
    if name == best_name:
        model_info['best_model'] = safe_name
    print(f'Saved {name} -> {path}')

with open('models/model_info.json', 'w') as f:
    json.dump(model_info, f, indent=2)

print(f'\nmodel_info.json written. Default model: {model_info["best_model"]}')


Best model: Random Forest (Tuned) (ROC-AUC = 0.9736)

Saved Logistic Regression -> models/logistic_regression.pkl
Saved Decision Tree -> models/decision_tree.pkl
Saved Random Forest -> models/random_forest.pkl
Saved SVM -> models/svm.pkl
Saved KNN -> models/knn.pkl
Saved Random Forest (Tuned) -> models/random_forest_tuned.pkl
Saved SVM (Tuned) -> models/svm_tuned.pkl
Saved KNN (Tuned) -> models/knn_tuned.pkl

model_info.json written. Default model: random_forest_tuned


## 11. Quick Verification

In [24]:
# Verification: reload best model and single-sample prediction
loaded = joblib.load(f'models/{model_info["best_model"]}.pkl')
sample = X_test.iloc[[0]]
display(sample)

pred = loaded.predict(sample)
prob = loaded.predict_proba(sample)[:, 1]
print(f"Prediction: {'Approved' if pred[0] == 1 else 'Rejected'}  (Prob: {prob[0]:.4f})")


,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file
10750,25.0,female,Bachelor,84973.0,2,MORTGAGE,14000.0,VENTURE,5.42,0.16,3.0,634,No


Prediction: Rejected  (Prob: 0.0150)
